# Section 6: EDA & Statistical Analysis

**Lead-to-Bank Ranking System** — Exploratory Data Analysis

This notebook covers all analyses required by Section 6 of the project spec:
1. Univariate distributions (continuous + categorical)
2. Log-normal fit verification (income, savings, loan_amount)
3. Bank profile analysis (min_cibil_score by bank_type; approval rate range)
4. Application analysis (conversion by bank_type; income_type × bank_type heatmap; rejection reasons)
5. Spearman correlation heatmap of LEAD_FEATURES
6. ALL_FEATURES vs `converted` correlation bar chart
7. VIF multicollinearity analysis
8. Validation of expected correlation directions

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import json

# Project imports
from src.features.feature_registry import (
    ALL_FEATURES, LEAD_FEATURES, BANK_FEATURES, INTERACTION_FEATURES,
    TEMPORAL_FEATURES, TARGET, GROUP_KEY
)

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

ARTIFACTS_DIR = Path('data/artifacts')
print(f'Working directory: {os.getcwd()}')
print(f'Artifacts directory: {ARTIFACTS_DIR}')

## 1. Data Loading

In [ ]:
leads = pd.read_parquet('data/raw/leads.parquet')
banks = pd.read_parquet('data/raw/banks.parquet')
apps_raw = pd.read_parquet('data/processed/applications_raw.parquet')
apps_feat = pd.read_parquet('data/processed/applications_features.parquet')

print('Dataset shapes:')
print(f'  leads:         {leads.shape}')
print(f'  banks:         {banks.shape}')
print(f'  apps_raw:      {apps_raw.shape}')
print(f'  apps_features: {apps_feat.shape}')
print()
print(f'Overall conversion rate: {apps_raw["converted"].mean():.4f} ({apps_raw["converted"].mean()*100:.2f}%)')
print(f'Eligibility pass rate:   {apps_raw["eligibility_passed"].mean():.4f}')

## 2. Univariate Analysis — Continuous Lead Features

In [ ]:
from src.eda.univariate import (
    plot_continuous_distributions,
    plot_log_normal_verification,
    verify_log_normal_skew,
    plot_categorical_distributions,
)

plot_continuous_distributions(leads, ARTIFACTS_DIR)
plt.figure(figsize=(15, 10))
img = plt.imread(str(ARTIFACTS_DIR / 'univariate_continuous_lead.png'))
plt.imshow(img)
plt.axis('off')
plt.title('Continuous Lead Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Log-Normal Fit Verification

> **Expected**: `annual_income`, `savings_balance`, `loan_amount_requested`, `credit_card_spend_monthly` should all be **right-skewed** (skewness > 0.5) and approximately log-normal.

In [ ]:
log_normal_stats = verify_log_normal_skew(leads)

print('Log-Normal Fit Results:')
print(f'{"Column":<35} {"Raw Skew":>12} {"Log Skew":>12} {"Right-Skewed?":>15} {"Status":>8}')
print('-' * 90)
for col, stats in log_normal_stats.items():
    status = '✓ PASS' if stats['is_right_skewed'] else '✗ FAIL'
    print(f'{col:<35} {stats["skewness"]:>12.3f} {stats["log_skewness"]:>12.3f} {str(stats["is_right_skewed"]):>15} {status:>8}')

In [ ]:
plot_log_normal_verification(leads, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'univariate_log_normal_fit.png'))
plt.figure(figsize=(12, 10))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Univariate Analysis — Categorical Features

In [ ]:
plot_categorical_distributions(leads, banks, ARTIFACTS_DIR)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, name in zip(axes, ['univariate_categorical_lead.png', 'univariate_categorical_bank.png']):
    img = plt.imread(str(ARTIFACTS_DIR / name))
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Key categorical summaries
print('Income type distribution:')
print(leads['income_type'].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print('Loan type distribution:')
print(leads['loan_type'].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print('Bank type distribution:')
print(banks['bank_type'].value_counts().to_string())

## 5. Bank Profile Analysis

> **CLAUDE.md §6**: min_cibil_score distribution by bank_type; approval rate range across banks.

In [ ]:
from src.eda.bivariate import plot_bank_profiles

plot_bank_profiles(banks, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'bivariate_bank_profiles.png'))
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Bank profile stats by type
bank_stats = banks.groupby('bank_type').agg(
    count=('bank_id', 'count'),
    min_cibil_mean=('min_cibil_score', 'mean'),
    min_cibil_std=('min_cibil_score', 'std'),
    approval_rate_mean=('approval_base_rate', 'mean'),
    approval_rate_std=('approval_base_rate', 'std'),
    disbursal_speed_mean=('disbursal_speed_days', 'mean'),
).round(2)

print('Bank Profile Statistics by Bank Type:')
display(bank_stats)

## 6. Application Analysis — Conversion Rates

In [ ]:
from src.eda.bivariate import (
    plot_conversion_by_bank_type,
    plot_conversion_heatmap,
    plot_rejection_reasons,
    plot_per_bank_conversion,
)

conversion_by_type = plot_conversion_by_bank_type(apps_feat, banks, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'bivariate_conversion_by_bank_type.png'))
plt.figure(figsize=(8, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print('Conversion rate by bank type:')
display(conversion_by_type)

In [ ]:
# income_type × bank_type conversion heatmap
pivot = plot_conversion_heatmap(apps_feat, leads, banks, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'bivariate_conversion_heatmap.png'))
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print('Conversion rate pivot (income_type × bank_type):')
display((pivot * 100).round(2))

In [ ]:
# Per-bank conversion std check (must be > 0.05)
per_bank = plot_per_bank_conversion(apps_feat, banks, ARTIFACTS_DIR)
per_bank_std = per_bank['conversion_rate'].std()

print(f'Per-bank conversion rate std: {per_bank_std:.4f} (threshold > 0.05)')
status = '✓ PASS' if per_bank_std > 0.05 else '✗ FAIL'
print(f'Status: {status}')

img = plt.imread(str(ARTIFACTS_DIR / 'bivariate_per_bank_conversion.png'))
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Rejection Reason Frequency

In [ ]:
rejection_summary = plot_rejection_reasons(apps_raw, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'bivariate_rejection_reasons.png'))
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print('Top rejection reasons:')
display(rejection_summary.head(10))

## 8. Spearman Correlation — Lead Features

In [ ]:
from src.eda.correlation_matrix import (
    plot_lead_feature_correlations,
    plot_feature_target_correlations,
    validate_correlation_directions,
    compute_vif,
    plot_vif,
)

corr_matrix = plot_lead_feature_correlations(apps_feat, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'correlation_lead_features.png'))
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Highlight strong correlations among lead features
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_unstacked = corr_matrix.mask(mask).unstack().dropna()
strong_corrs = corr_unstacked[abs(corr_unstacked) > 0.5].sort_values(key=abs, ascending=False)

print('Strong Spearman correlations (|rho| > 0.50) among LEAD_FEATURES:')
print(strong_corrs.to_string())

## 9. ALL_FEATURES vs `converted` Correlation

In [ ]:
corrs = plot_feature_target_correlations(apps_feat, ARTIFACTS_DIR)
img = plt.imread(str(ARTIFACTS_DIR / 'correlation_features_vs_target.png'))
plt.figure(figsize=(10, max(8, len(corrs) * 0.28)))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print('Top 10 features positively correlated with converted:')
print(corrs[corrs > 0].head(10).round(4).to_string())
print()
print('Top 10 features negatively correlated with converted:')
print(corrs[corrs < 0].head(10).round(4).to_string())

## 10. Expected Correlation Direction Validation

> All 6 expected directions must hold per CLAUDE.md §6.

In [ ]:
direction_results = validate_correlation_directions(apps_feat)

print(f'{"Feature":<30} {"Expected":>12} {"Actual":>12} {"Corr":>10} {"Status":>8}')
print('-' * 80)
all_pass = True
for feat, res in direction_results.items():
    status = res.get('status', 'SKIP')
    if status == 'FAIL':
        all_pass = False
    icon = '✓' if status == 'PASS' else ('✗' if status == 'FAIL' else '-')
    print(f'{feat:<30} {res.get("expected", "-"):>12} {res.get("actual", "-"):>12} {res.get("corr", 0):>10.4f} {icon + " " + status:>8}')

print()
if all_pass:
    print('✓ ALL correlation direction checks PASSED')
else:
    print('✗ Some correlation direction checks FAILED — review simulation logic')

## 11. VIF — Multicollinearity Analysis

In [ ]:
vif_data = compute_vif(apps_feat)
plot_vif(vif_data, ARTIFACTS_DIR)

img = plt.imread(str(ARTIFACTS_DIR / 'correlation_vif.png'))
plt.figure(figsize=(9, max(5, len(vif_data) * 0.35)))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print('VIF Analysis Summary:')
print(f'  Features with VIF > 10: {len(vif_data[vif_data["VIF"] > 10])}')
print(f'  Features with VIF > 5:  {len(vif_data[vif_data["VIF"] > 5])}')
print(f'  Max VIF: {vif_data["VIF"].max():.1f}')
print()
print('High VIF features (> 10):')
display(vif_data[vif_data['VIF'] > 10].reset_index(drop=True))

In [ ]:
# VIF interpretation — high VIF is expected for engineered features
print('VIF Interpretation Notes:')
print('- age_at_maturity = age + loan_tenure_months/12 → expected high VIF with age')
print('- cibil_score ↔ min_cibil_score → cibil_gap derived from both')
print('- interest_rate_min ↔ interest_rate_max → by design, narrow range per bank')
print('- annual_income ↔ credit_card_spend_monthly → correlated income proxies')
print('- These are known causal relationships, not spurious correlations')
print('- XGBoost handles multicollinearity well via random feature subsampling (colsample_bytree)')

## 12. Simulation Realism Assertions

> All assertions from CLAUDE.md §7 must pass.

In [ ]:
# Sample for fast computation
sample = apps_feat.sample(50_000, random_state=42)

checks = {
    'conversion_rate_in_range': 0.10 <= apps_feat['converted'].mean() <= 0.22,
    'per_bank_conversion_std_gt_0.05': apps_feat.groupby('bank_id')['converted'].mean().std() > 0.05,
    'cibil_income_corr_gt_0.30': (
        leads['cibil_score'].corr(leads['annual_income'], method='spearman') > 0.30
    ),
    'cibil_dpd30_corr_lt_neg0.20': (
        leads['cibil_score'].corr(leads['dpd_30_count'], method='spearman') < -0.20
    ),
    'foir_headroom_corr_gt_0.10': (
        sample['foir_headroom'].corr(sample['converted'], method='spearman') > 0.10
    ),
    'bureau_fatigue_corr_lt_neg0.05': (
        sample['bureau_fatigue_flag'].corr(sample['converted'], method='spearman') < -0.05
    ),
}

print('Simulation Realism Assertions:')
all_pass = True
for name, result in checks.items():
    icon = '✓' if result else '✗'
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass = False
    print(f'  {icon} {name}: {status}')

# Print actual values
print()
print('Actual values:')
print(f'  Conversion rate:           {apps_feat["converted"].mean():.4f}')
print(f'  Per-bank conversion std:   {apps_feat.groupby("bank_id")["converted"].mean().std():.4f}')
print(f'  CIBIL-income corr:         {leads["cibil_score"].corr(leads["annual_income"], method="spearman"):.4f}')
print(f'  CIBIL-DPD30 corr:          {leads["cibil_score"].corr(leads["dpd_30_count"], method="spearman"):.4f}')
print(f'  FOIR-headroom-converted:   {sample["foir_headroom"].corr(sample["converted"], method="spearman"):.4f}')
print(f'  Bureau fatigue-converted:  {sample["bureau_fatigue_flag"].corr(sample["converted"], method="spearman"):.4f}')

if all_pass:
    print()
    print('✓ ALL simulation realism assertions PASSED')
else:
    print()
    print('✗ Some assertions FAILED — review simulation or feature engineering')

## 13. Feature Group Coverage Summary

In [ ]:
feat_cols = [c for c in ALL_FEATURES if c in apps_feat.columns]
missing_feats = [c for c in ALL_FEATURES if c not in apps_feat.columns]

print('Feature Coverage in applications_features.parquet:')
print(f'  LEAD_FEATURES:        {sum(c in apps_feat.columns for c in LEAD_FEATURES)}/{len(LEAD_FEATURES)}')
print(f'  BANK_FEATURES:        {sum(c in apps_feat.columns for c in BANK_FEATURES)}/{len(BANK_FEATURES)}')
print(f'  INTERACTION_FEATURES: {sum(c in apps_feat.columns for c in INTERACTION_FEATURES)}/{len(INTERACTION_FEATURES)}')
print(f'  TEMPORAL_FEATURES:    {sum(c in apps_feat.columns for c in TEMPORAL_FEATURES)}/{len(TEMPORAL_FEATURES)}')
print(f'  TOTAL:                {len(feat_cols)}/{len(ALL_FEATURES)}')
if missing_feats:
    print(f'  Missing: {missing_feats}')

In [ ]:
# Null count check
null_counts = apps_feat[feat_cols].isnull().sum()
if null_counts.sum() == 0:
    print('✓ Zero null values in feature matrix')
else:
    print('✗ Null values found:')
    print(null_counts[null_counts > 0].to_string())

## 14. EDA Summary & Key Findings

In [ ]:
with open(ARTIFACTS_DIR / 'eda_summary.json') as f:
    eda_summary = json.load(f)

print('=== EDA SUMMARY ===')
print()
print('UNIVARIATE:')
for col, stats in eda_summary['univariate']['log_normal_stats'].items():
    print(f'  {col}: skew={stats["skewness"]:.1f}, right_skewed={stats["is_right_skewed"]}')

print()
print('BIVARIATE:')
print(f'  Overall conversion rate: {eda_summary["bivariate"]["overall_conversion_rate"]:.4f}')
print(f'  Per-bank std: {eda_summary["bivariate"]["per_bank_std"]:.4f}')
print('  Top 5 rejection reasons:')
for r in eda_summary['bivariate']['top_rejection_reasons']:
    print(f'    {r["reason"]}: {r["n"]:,} ({r["pct"]:.1f}%)')

print()
print('CORRELATION:')
print('  Direction checks:')
for feat, res in eda_summary['correlation']['direction_checks'].items():
    print(f'    {feat}: {res.get("status", "SKIP")} (corr={res.get("corr", 0):.4f})')

if eda_summary['correlation']['high_vif_features']:
    print(f'  High VIF features: {eda_summary["correlation"]["high_vif_features"]}')
else:
    print('  No high VIF features detected')

## 15. ydata-profiling HTML Report

The full interactive HTML report was generated at `data/artifacts/data_report.html`.
Open it in a browser for detailed per-feature stats, correlations, and missing value analysis.

In [ ]:
report_path = ARTIFACTS_DIR / 'data_report.html'
if report_path.exists():
    print(f'✓ HTML report available at: {report_path.resolve()}')
    print(f'  File size: {report_path.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('HTML report not yet generated. Run:')
    print('  python -m src.eda.report_generator')